## Step 0.1 — Environment and Library Versions

In [11]:
import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)

3.13.1 2.3.4 2.3.3 1.7.2 3.10.7


## Step 0.2 — Dataset Loading and Train-Test Split

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 3113

data = pd.read_csv("data/spambase.csv")

feature_names = [c for c in data.columns if c != "spam"]

X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=SEED
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)

(4204, 57) 39.9 (3153, 57) (1051, 57)


The dataset contains 4,204 emails and 57 numerical features. About 39.9% of the emails are classified as spam. The data was divided into a training set with 3,153 emails (75%) and a test set with 1,051 emails (25%).

## Step 0.3 — Model Definitions

In [13]:
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)

bag = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=100,
    oob_score=True,
    random_state=SEED
)

rf = RandomForestClassifier(
    n_estimators=300,
    oob_score=True,
    random_state=SEED,
    n_jobs=-1
)

lr = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

bag_lr = BaggingClassifier(
    lr,
    n_estimators=50,
    random_state=SEED
)

knn = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=15)
)

ada = AdaBoostClassifier(
    n_estimators=400,
    random_state=SEED
)

members = [("lr", lr), ("knn", knn), ("tree", tree)]

print("All models have been defined successfully.")

All models have been defined successfully.


# Part 1 — Bagging and Out-of-Bag Estimates 

## Step 1.1 — Implementing the OOB Mask Function

In [15]:
def oob_mask(sample_indices, n_samples):
    """Return a boolean mask of samples not included in the bootstrap sample."""
    mask = np.ones(n_samples, dtype=bool)
    mask[np.asarray(sample_indices, dtype=int)] = False
    return mask



print(oob_mask([0, 2, 2], 5))
print(oob_mask([], 3))
print(oob_mask(np.array([3, 0]), 4))

[False  True False  True  True]
[ True  True  True]
[False  True  True False]




The `oob_mask` function identifies which training samples were not selected in a bootstrap sample. It returns `True` for samples that were not used and `False` for samples that were selected.



## Step 1.2 — Out-of-Bag Share Calculation

In [16]:

bag.fit(X_train, y_train)

n = len(X_train)

oob_shares = []

for sample_indices in bag.estimators_samples_:
    mask = oob_mask(sample_indices, n)
    oob_shares.append(mask.mean())

# Average shares
mean_oob_share = np.mean(oob_shares)
mean_inbag_share = 1 - mean_oob_share

# Theoretical probabilities
theoretical_oob = (1 - 1 / n) ** n
theoretical_inbag = 1 - theoretical_oob

print(f"Number of training samples: {n}")
print(f"Number of trees: {len(bag.estimators_)}")

print(f"\nMean OOB share: {mean_oob_share:.4f}")
print(f"Mean in-bag share: {mean_inbag_share:.4f}")

print(f"\nTheoretical OOB share: {theoretical_oob:.4f}")
print(f"Theoretical in-bag share: {theoretical_inbag:.4f}")

print(f"\nDifference: {abs(mean_oob_share - theoretical_oob):.6f}")

Number of training samples: 3153
Number of trees: 100

Mean OOB share: 0.3680
Mean in-bag share: 0.6320

Theoretical OOB share: 0.3678
Theoretical in-bag share: 0.6322

Difference: 0.000133


## Step 1.3 — Calculating OOB Accuracy Manually

In [17]:

n = len(X_train)
votes = np.zeros((n, 2), dtype=int)


for tree_model, sample_indices in zip(
    bag.estimators_, bag.estimators_samples_
):
    mask = oob_mask(sample_indices, n)

    predictions = tree_model.predict(X_train[mask])

    row_indices = np.where(mask)[0]
    for row, prediction in zip(row_indices, predictions):
        votes[row, int(prediction)] += 1


oob_predictions = np.argmax(votes, axis=1)

valid_rows = votes.sum(axis=1) > 0
manual_oob_accuracy = np.mean(
    oob_predictions[valid_rows] == y_train[valid_rows]
)

print(f"Manual OOB accuracy: {manual_oob_accuracy:.4f}")
print(f"Sklearn OOB accuracy: {bag.oob_score_:.4f}")
print(f"Difference: {abs(manual_oob_accuracy - bag.oob_score_):.6f}")
print(f"Rows with no OOB votes: {(~valid_rows).sum()}")

Manual OOB accuracy: 0.9375
Sklearn OOB accuracy: 0.9375
Difference: 0.000000
Rows with no OOB votes: 0


## Step 1.4 — Five-Fold Cross-Validation

In [8]:
from sklearn.model_selection import cross_validate


models = {
    "Decision Tree": tree,
    "Bagging Tree": bag,
    "Random Forest": rf,
    "Logistic Regression": lr,
    "Bagging LR": bag_lr
}


cv_scores = {}

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )["test_score"]

    cv_scores[name] = scores


results = []

for name, scores in cv_scores.items():
    results.append({
        "Model": name,
        "Fold 1": scores[0],
        "Fold 2": scores[1],
        "Fold 3": scores[2],
        "Fold 4": scores[3],
        "Fold 5": scores[4],
        "Mean": np.mean(scores),
        "Std": np.std(scores, ddof=0)
    })

cv_table = pd.DataFrame(results)

display(cv_table.round(4))

,Model,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean,Std
0,Decision Tree,0.8875,0.9049,0.9128,0.8968,0.9143,0.9033,0.0101
1,Bagging Tree,0.9445,0.9319,0.9271,0.9302,0.9429,0.9353,0.0070
2,Random Forest,0.9540,0.9429,0.9382,0.9444,0.9508,0.9461,0.0057
3,Logistic Regression,0.9477,0.9429,0.9366,0.9365,0.9444,0.9416,0.0044
4,Bagging LR,0.9477,0.9398,0.9350,0.9413,0.9444,0.9416,0.0043


## Step 1.5 — Paired Fold Differences

In [10]:

comparisons = {
    "Bagging Tree - Decision Tree":
        cv_scores["Bagging Tree"] - cv_scores["Decision Tree"],

    "Random Forest - Decision Tree":
        cv_scores["Random Forest"] - cv_scores["Decision Tree"],

    "Bagging LR - Logistic Regression":
        cv_scores["Bagging LR"] - cv_scores["Logistic Regression"]
}

difference_results = []

for name, differences in comparisons.items():
    mean_diff = np.mean(differences)
    std_diff = np.std(differences, ddof=0)

    if abs(mean_diff) <= std_diff:
        conclusion = "Inside fold spread (no clear winner)"
    elif mean_diff > 0:
        conclusion = "First model performs better"
    else:
        conclusion = "Second model performs better"

    difference_results.append({
        "Comparison": name,
        "Fold 1": differences[0],
        "Fold 2": differences[1],
        "Fold 3": differences[2],
        "Fold 4": differences[3],
        "Fold 5": differences[4],
        "Mean Difference": mean_diff,
        "Std Difference": std_diff,
        "Conclusion": conclusion
    })

difference_table = pd.DataFrame(difference_results)

display(difference_table.round(4))

,Comparison,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean Difference,Std Difference,Conclusion
0,Bagging Tree - Decision Tree,0.0571,0.0269,0.0143,0.0333,0.0286,0.0320,0.0140,First model performs better
1,Random Forest - Decision Tree,0.0666,0.0380,0.0254,0.0476,0.0365,0.0428,0.0138,First model performs better
2,Bagging LR - Logistic Regression,0.0000,-0.0032,-0.0016,0.0048,0.0000,0.0000,0.0027,Inside fold spread (no clear winner)


## Step 1.6 — Random Forest: OOB, CV and Test Accuracy

In [18]:
from sklearn.metrics import accuracy_score


rf.fit(X_train, y_train)

rf_oob = rf.oob_score_

rf_cv_scores = cv_scores["Random Forest"]
rf_cv_mean = np.mean(rf_cv_scores)
rf_cv_std = np.std(rf_cv_scores, ddof=0)

rf_test_predictions = rf.predict(X_test)
rf_test_accuracy = accuracy_score(y_test, rf_test_predictions)


n_test = len(y_test)
standard_error = np.sqrt(
    rf_test_accuracy * (1 - rf_test_accuracy) / n_test
)


difference = abs(rf_oob - rf_cv_mean)
agreement = difference <= rf_cv_std

print(f"Random Forest OOB accuracy: {rf_oob:.4f}")
print(f"Random Forest CV mean accuracy: {rf_cv_mean:.4f}")
print(f"Random Forest CV std: {rf_cv_std:.4f}")
print(f"Random Forest test accuracy: {rf_test_accuracy:.4f}")
print(f"Test standard error: {standard_error:.4f}")

print(f"\nOOB-CV difference: {difference:.4f}")
print(f"OOB and CV agree: {agreement}")

Random Forest OOB accuracy: 0.9486
Random Forest CV mean accuracy: 0.9461
Random Forest CV std: 0.0057
Random Forest test accuracy: 0.9429
Test standard error: 0.0072

OOB-CV difference: 0.0025
OOB and CV agree: True


### Interpretation

The Random Forest model achieved an OOB accuracy of 94.86%, a mean cross-validation accuracy of 94.61%, and a test accuracy of 94.29%. These results are close to each other, which suggests that the model performs consistently on different data.

The standard deviation of the cross-validation scores was 0.0057, while the standard error of the test accuracy was 0.0072.

The difference between OOB and CV accuracy was 0.0025, which is smaller than the CV standard deviation of 0.0057. Therefore, the OOB and CV results are in agreement.

Overall, OOB accuracy is a reasonable estimate of cross-validation performance for this Random Forest model. The test accuracy is slightly lower, but the difference is small compared with its standard error.